# 05 - Empirical Analysis & Primary Thesis Publication Figures

**Active Primary Evaluation Suite: Cross-Model Scaling, Prompt Scaffolding & Multi-Noise Factorial Benchmarks**

This notebook serves as the primary publication analysis pipeline for the master thesis, systematically generating high-resolution empirical figures:
1. **Section 1: Environment Setup & Data Ingestion**: System configuration, paths, trace loading, and adaptive target matrix preparation.
2. **Section 2: Core Thesis Publication Figures**:
   - **Figure 9D (RQ1)**: Full Factorial Solver × BBOB Problem Function Performance Matrix ().
   - **Figure 9E (Sub-RQ A)**: LLM Parameter Scale Scaling Laws Across Search Dimensions ().
   - **Landscape Hardness Success Profiles**: Success rates across separable, conditioning, multi-modal, and deceptive classes.
3. **Section 3: Search Trajectories & Empirical Runtime ECDFs**:
   - **3.1 Part 1: Explicit Noise Performance Profiles**: Noise-informed LLaMEA champions vs. Classical Baselines ().
   - **3.2 Part 2: Implicit Noise Performance Profiles**: Noise-blind LLaMEA heuristics deployed under noise vs. Classical Baselines ().
   - **3.3 Part 3: Implicit vs. Explicit Comparison**: Head-to-head comparison of informed vs. blind adaptation ().
   - **3.4 Part 4: Single-Algorithm Cross-Environment Noise Overlays**: Generalization and robustness degradation across $\sigma \in \{0.0, 0.05, 0.1, 0.2\}$ ().

*(Note: Exploratory inferential testing, A12 heatmaps, benchmark difficulty shift, and failure diagnostics have been archived in [](05_legacy_figures.ipynb)).*

---
## Section 1: Environment Setup & Data Ingestion

### 1.1 Environment Setup, Paths & Plotly Theme Initialization
Initializes reporting directories (`results/main_results`, `results/profiles`, `results/failure_analysis`, etc.), service dependencies, and typography standards.

In [1]:
%load_ext autoreload
%autoreload 2

# Ensure project root src/ is in sys.path
import sys, os, re, json, math, time
from pathlib import Path
cwd = Path(".").resolve()
src_dir = cwd.parent / "src" if cwd.name == "notebooks" else cwd / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from collections import defaultdict
import colorsys
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Configure Plotly / Kaleido performance: disable slow MathJax scanner
import plotly.io as pio
pio.defaults.mathjax = None
if hasattr(pio, "kaleido") and hasattr(pio.kaleido, "scope"):
    pio.kaleido.scope.mathjax = None

# Base directories & Output paths
from shared.config import RESULTS_DIR

LANDSCAPES_DIR           = RESULTS_DIR / "figures" / "01_landscapes"
EXPLICIT_DIR             = RESULTS_DIR / "figures" / "02_explicit"
IMPLICIT_DIR             = RESULTS_DIR / "figures" / "03_implicit"
IMPLICIT_VS_EXPLICIT_DIR = RESULTS_DIR / "figures" / "04_implicit_vs_explicit"
CROSS_EVAL_DIR           = RESULTS_DIR / "figures" / "05_cross_evaluation"
THESIS_SUMMARY_DIR       = RESULTS_DIR / "figures" / "06_thesis_summary"
REPORTS_DIR              = RESULTS_DIR / "reports"
TRACES_DIR               = RESULTS_DIR / "ioh_traces"
EVALUATIONS_DIR          = TRACES_DIR

# Backwards compatibility aliases
PROFILES_DIR             = EXPLICIT_DIR
IMPLICIT_COMPARISON_DIR  = IMPLICIT_VS_EXPLICIT_DIR
MAIN_RESULTS_DIR         = THESIS_SUMMARY_DIR

for d in [LANDSCAPES_DIR, EXPLICIT_DIR, IMPLICIT_DIR, IMPLICIT_VS_EXPLICIT_DIR, CROSS_EVAL_DIR, THESIS_SUMMARY_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Global toggle to force re-exporting of static figures
FORCE_RERUN = False

def is_figure_outdated(target_file: Path, condition_trace_dirs: list[Path] | None = None, force: bool = False) -> bool:
    """Smart cache checker: returns True if target figure does not exist or source traces are newer."""
    if force or FORCE_RERUN:
        return True
    if not os.path.exists(target_file):
        return True
    if not condition_trace_dirs:
        return False
    try:
        target_mtime = os.path.getmtime(target_file)
        for tdir in condition_trace_dirs:
            if not os.path.exists(tdir):
                continue
            prov_file = os.path.join(tdir, "evaluation_provenance.json")
            if os.path.exists(prov_file) and os.path.getmtime(prov_file) > target_mtime:
                return True
            if os.path.getmtime(tdir) > target_mtime:
                return True
        return False
    except Exception:
        return True

# Initialize domain services
from shared.database.engine import create_db_session_factory
from benchmarking.infra.storage import SQLiteSynthesisReadRepository
from benchmarking.infra.io.trace_repository import IOHTraceReader
from benchmarking.application.statistical_service import StatisticalEvaluationService
from benchmarking.domain.enums import BBOBFunction
from benchmarking.domain import EvaluationCondition, EvaluationDataset, RunTrace
from benchmarking.domain.services.resolvers import (
    resolve_folder_solver_name,
    resolve_canonical_model_slug,
    get_clean_model_label,
    get_model_slug,
)

PROBLEM_IDS = [1, 8, 11, 15, 21]
CLASSICAL_BASELINES = ["cma-es", "pso", "de"]

session_factory = create_db_session_factory()
sqlite_repo = SQLiteSynthesisReadRepository(session_factory)
trace_reader = IOHTraceReader()

def _get_conv(self, all_benchmark_data, dim, noise_std, problem_id, solver, eval_grid):
    cond = EvaluationCondition(dim=dim, noise_std=noise_std, problem_id=problem_id)
    runs = all_benchmark_data.get(cond, {}).get(solver, [])
    if not runs:
        for k, v in all_benchmark_data.get(cond, {}).items():
            if str(k) == str(solver) or getattr(k, 'value', None) == str(solver):
                runs = v
                break
    if not runs:
        return None
    targets = self.compute_adaptive_targets(all_benchmark_data, noise_std=noise_std)
    mean_t, q25_t, q75_t, _ = self.compute_trajectory_and_ecdf(runs, eval_grid, targets)
    return {"median": mean_t, "q25": q25_t, "q75": q75_t}

def _get_agg_conv(self, all_benchmark_data, dim, noise_std, solver, eval_grid):
    meds, q25s, q75s = [], [], []
    for pid in all_benchmark_data.problem_ids:
        r = self.get_convergence_trajectory(all_benchmark_data, dim, noise_std, pid, solver, eval_grid)
        if r is not None:
            meds.append(r["median"])
            q25s.append(r["q25"])
            q75s.append(r["q75"])
    if not meds:
        return None
    return {"median": np.nanmean(meds, axis=0), "q25": np.nanmean(q25s, axis=0), "q75": np.nanmean(q75s, axis=0)}

def _get_ecdf(self, all_benchmark_data, target_evals=None, dim=None, noise_std=None, problem_id=None, solver=None, eval_grid=None, targets=None):
    cond = EvaluationCondition(dim=dim, noise_std=noise_std, problem_id=problem_id)
    runs = all_benchmark_data.get(cond, {}).get(solver, [])
    if not runs:
        for k, v in all_benchmark_data.get(cond, {}).items():
            if str(k) == str(solver) or getattr(k, 'value', None) == str(solver):
                runs = v
                break
    if not runs:
        return None
    actual_targets = target_evals if target_evals is not None else (targets if targets is not None else self.compute_adaptive_targets(all_benchmark_data, noise_std=noise_std))
    _, _, _, ecdf_t = self.compute_trajectory_and_ecdf(runs, eval_grid, actual_targets)
    return ecdf_t

def _get_agg_ecdf(self, all_benchmark_data, target_evals=None, dim=None, noise_std=None, solver=None, eval_grid=None, targets=None):
    ecdfs = []
    for pid in all_benchmark_data.problem_ids:
        e = self.get_target_ecdf_curve(all_benchmark_data, target_evals, dim=dim, noise_std=noise_std, problem_id=pid, solver=solver, eval_grid=eval_grid)
        if e is not None:
            ecdfs.append(e)
    if not ecdfs:
        return None
    return np.nanmean(ecdfs, axis=0)

StatisticalEvaluationService.get_convergence_trajectory = _get_conv
StatisticalEvaluationService.get_aggregate_convergence = _get_agg_conv
StatisticalEvaluationService.get_target_ecdf_curve = _get_ecdf
StatisticalEvaluationService.get_aggregate_target_ecdf_curve = _get_agg_ecdf

service = StatisticalEvaluationService(sqlite_repo=sqlite_repo, trace_repo=trace_reader)
class ThesisPaletteAdapter:
    def get_color(self, solver_name: str) -> str:
        return get_solver_color(str(solver_name))
    def get_dash(self, solver_name: str) -> str:
        return get_solver_line_style(str(solver_name)).get("dash", "solid")
    def clean_name(self, solver_name: str) -> str:
        s = str(solver_name)
        for rm in [" (noise-adapted)", " (noise-implicit)"]:
            s = s.replace(rm, "")
        return s.strip()
    def hex_to_rgba(self, hex_color: str, opacity: float = 0.12) -> str:
        return get_rgba_fill(hex_color, opacity)

service.palette = ThesisPaletteAdapter()

# ─── Typography & Dynamic Visualization Palette Engine ───────────────────────
FONT_FAMILY = "Inter, -apple-system, BlinkMacSystemFont, Arial, sans-serif"

STRATEGY_COLOR_ARCHETYPES = {
    "guided": {"large": "#38BDF8", "small": "#38BDF8", "base": "#38BDF8"},
    "thinking": {"large": "#34D399", "small": "#34D399", "base": "#34D399"},
    "vectorization": {"large": "#F87171", "small": "#F87171", "base": "#F87171"},
    "baseline": {"large": "#FBBF24", "small": "#FBBF24", "base": "#FBBF24"},
}
STRATEGY_PALETTE = {k: v["base"] for k, v in STRATEGY_COLOR_ARCHETYPES.items()}

CLASSICAL_SOLVERS_STYLE = {
    "cma-es": {"color": "#334155", "dash": "dash", "width": 2.2, "name": "CMA-ES"},
    "pso": {"color": "#0D9488", "dash": "dashdot", "width": 2.2, "name": "PSO"},
    "de": {"color": "#7C3AED", "dash": "dot", "width": 2.2, "name": "DE"},
}

REGIME_PALETTE = {
    "clean": {"color": "#38BDF8", "border": "rgba(15, 23, 42, 0.4)", "name": "Clean (σ=0.0)", "pattern": None},
    "noisy": {"color": "#FB923C", "border": "rgba(124, 45, 18, 0.4)", "name": "Noisy (σ=0.05)", "pattern": {"shape": "/", "fillmode": "replace", "fgcolor": "#FFFFFF", "fgopacity": 0.35, "size": 6}},
}

DIMENSION_PALETTE_CLEAN = {2: "#BAE6FD", 3: "#7DD3FC", 5: "#38BDF8", 10: "#0284C7"}
DIMENSION_PALETTE_NOISY = {2: "#FED7AA", 3: "#FDBA74", 5: "#FB923C", 10: "#F97316"}
MODEL_SCALE_PALETTE = {"Qwen2.5-Coder-3B": "#BAE6FD", "Qwen2.5-Coder-7B": "#38BDF8", "Qwen2.5-Coder-14B": "#0284C7", "Qwen2.5-Coder-32B": "#0369A1"}

def get_model_scale_color(model_name: str) -> str:
    m_clean = model_name.strip()
    for k, color in MODEL_SCALE_PALETTE.items():
        if k.lower() in m_clean.lower():
            return color
    if "32b" in m_clean.lower():
        return "#0369A1"
    elif "14b" in m_clean.lower():
        return "#0284C7"
    elif "7b" in m_clean.lower():
        return "#38BDF8"
    elif "3b" in m_clean.lower():
        return "#BAE6FD"
    return "#0284C7"

def hsl_to_hex(h: float, s: float, lightness: float) -> str:
    r, g, b = colorsys.hls_to_rgb(h, lightness, s)
    return "#{:02X}{:02X}{:02X}".format(
        int(round(max(0.0, min(1.0, r)) * 255)),
        int(round(max(0.0, min(1.0, g)) * 255)),
        int(round(max(0.0, min(1.0, b)) * 255)),
    )

_DYNAMIC_STYLE_CACHE = {}

def get_solver_line_style(solver_name: str) -> dict:
    if not solver_name:
        return {"color": "#64748B", "dash": "solid", "width": 2.0}
    s = solver_name.strip()
    s_lower = s.lower()
    if s in _DYNAMIC_STYLE_CACHE:
        return _DYNAMIC_STYLE_CACHE[s]
    if s_lower in CLASSICAL_SOLVERS_STYLE:
        res = CLASSICAL_SOLVERS_STYLE[s_lower].copy()
        _DYNAMIC_STYLE_CACHE[s] = res
        return res
    if " / " in s:
        import re
        model_part, strat_part = s.split(" / ", 1)
        strat_key = strat_part.strip().lower()
        size_match = re.search(r"(\d+(?:\.\d+)?)\s*[bB]", model_part)
        is_large = float(size_match.group(1)) >= 14.0 if size_match else True
        scale_key = "large" if is_large else "small"
        line_width = 2.5 if is_large else 1.8
        if strat_key in STRATEGY_COLOR_ARCHETYPES:
            hex_color = STRATEGY_COLOR_ARCHETYPES[strat_key][scale_key]
        else:
            base_hue = (abs(hash(strat_key)) * 0.618033988749895) % 1.0
            hex_color = hsl_to_hex(base_hue, s=0.85, lightness=0.45 if is_large else 0.62)
        res = {"color": hex_color, "dash": "solid", "width": line_width}
        _DYNAMIC_STYLE_CACHE[s] = res
        return res
    hue = (abs(hash(s_lower)) * 0.618033988749895) % 1.0
    hex_color = hsl_to_hex(hue, s=0.75, lightness=0.50)
    res = {"color": hex_color, "dash": "dash", "width": 2.0}
    _DYNAMIC_STYLE_CACHE[s] = res
    return res

def get_solver_color(solver_name: str) -> str:
    return get_solver_line_style(solver_name)["color"]

def get_rgba_fill(hex_color: str, opacity: float = 0.12) -> str:
    if hex_color.startswith("#") and len(hex_color) == 7:
        r = int(hex_color[1:3], 16)
        g = int(hex_color[3:5], 16)
        b = int(hex_color[5:7], 16)
        return f"rgba({r}, {g}, {b}, {opacity})"
    return f"rgba(100, 116, 139, {opacity})"

def get_dimension_color(dim: int, is_noisy: bool = False) -> str:
    if is_noisy:
        return DIMENSION_PALETTE_NOISY.get(dim, "#FB923C")
    return DIMENSION_PALETTE_CLEAN.get(dim, "#3B82F6")

def build_dynamic_solver_palette(solvers) -> dict:
    return {s: get_solver_color(s) for s in solvers}

class DynamicSolverPalette(dict):
    def __getitem__(self, key: str) -> str:
        return get_solver_color(key)

    def get(self, key, default=None):
        if not key:
            return default or "#64748B"
        return get_solver_color(str(key))

def get_noise_color(noise_std: float, all_stds: list[float] | None = None) -> str:
    """Returns an eye-friendly soft color for any given noise level."""
    if noise_std == 0.0:
        return "#38BDF8"
    if all_stds is None or len(all_stds) <= 2:
        if np.isclose(noise_std, 0.05):
            return "#FB923C"
        elif np.isclose(noise_std, 0.1):
            return "#F87171"
        elif np.isclose(noise_std, 0.2):
            return "#C084FC"
        return "#FB923C"
    noisy_stds = sorted([s for s in all_stds if s > 0.0])
    if not noisy_stds:
        return "#FB923C"
    idx = noisy_stds.index(noise_std) if noise_std in noisy_stds else 0
    palette = ["#FDBA74", "#FB923C", "#F87171", "#FB7185", "#E879F9", "#C084FC", "#A78BFA"]
    return palette[idx % len(palette)]

# Factorial filtering constraints
FILTER_DIMS = None
FILTER_PROBLEMS = None
FILTER_NOISE_STDS = None

print("✅ Analysis environment, domain paths, and dynamic styling engine initialized.")


✅ Statistical service and unified dynamic thesis visualization palette initialized.


### 1.2 Ingest Benchmark Traces & Synthesis Database Records
Loads all completed evolutionary synthesis experiments from SQLite (`data/db.sqlite3`) and all empirical evaluation traces ($N=20$) from `results/ioh_traces/`.

In [2]:
# Ingest benchmark traces and synthesis records
df_exp, df_iter = service.get_synthesis_dataframes()
all_benchmark_data = service.load_evaluation_traces(
    dims=FILTER_DIMS,
    problems=FILTER_PROBLEMS,
    noise_stds=FILTER_NOISE_STDS,
    solver_resolver=resolve_folder_solver_name,
)

if not all_benchmark_data:
    raise RuntimeError(f'No benchmark traces found in {EVALUATIONS_DIR}!')

all_dims = all_benchmark_data.dims
all_noise_stds = all_benchmark_data.noise_stds
clean_std = 0.0 if 0.0 in all_noise_stds else (all_noise_stds[0] if all_noise_stds else 0.0)
noisy_std = next((n for n in all_noise_stds if n > 0.0), all_noise_stds[-1] if all_noise_stds else 0.05)
PROBLEM_IDS = all_benchmark_data.problem_ids

DISCOVERED_SOLVERS = all_benchmark_data.solvers
SOLVER_PALETTE = build_dynamic_solver_palette(DISCOVERED_SOLVERS)

MODELS_TO_SOLVERS = defaultdict(list)
for s in DISCOVERED_SOLVERS:
    if ' / ' in s:
        MODELS_TO_SOLVERS[s.split(' / ')[0]].append(s)

LLM_SOLVERS_ORDER = [s for s in DISCOVERED_SOLVERS if ' / ' in s]
CLASSICAL_SOLVERS_ORDER = [s for s in DISCOVERED_SOLVERS if ' / ' not in s]
ALL_SOLVERS_ORDER = LLM_SOLVERS_ORDER + CLASSICAL_SOLVERS_ORDER

print(f'📦 Loaded {len(df_exp)} experiments and {len(all_benchmark_data)} problem conditions.')
print(f'🎯 Problems: {PROBLEM_IDS} | Dimensions: {all_dims} | Solvers: {DISCOVERED_SOLVERS}')


📦 Loaded 3550 experiments and 80 problem conditions.
🎯 Problems: [1, 8, 11, 15, 21] | Dimensions: [2, 3, 5, 10] | Solvers: [<ClassicalSolver.CMA_ES: 'CMA-ES'>, <ClassicalSolver.DE: 'DE'>, <ClassicalSolver.PSO: 'PSO'>, 'Qwen2.5-Coder-14B / baseline', 'Qwen2.5-Coder-14B / baseline (noise-adapted)', 'Qwen2.5-Coder-14B / baseline (noise-implicit)', 'Qwen2.5-Coder-14B / guided', 'Qwen2.5-Coder-14B / guided (noise-adapted)', 'Qwen2.5-Coder-14B / guided (noise-implicit)', 'Qwen2.5-Coder-14B / thinking', 'Qwen2.5-Coder-14B / thinking (noise-adapted)', 'Qwen2.5-Coder-14B / thinking (noise-implicit)', 'Qwen2.5-Coder-14B / vectorization', 'Qwen2.5-Coder-14B / vectorization (noise-adapted)', 'Qwen2.5-Coder-14B / vectorization (noise-implicit)', 'Qwen2.5-Coder-32B / baseline', 'Qwen2.5-Coder-32B / baseline (noise-adapted)', 'Qwen2.5-Coder-32B / baseline (noise-implicit)', 'Qwen2.5-Coder-32B / guided', 'Qwen2.5-Coder-32B / guided (noise-adapted)', 'Qwen2.5-Coder-32B / guided (noise-implicit)', 'Qwen

### 1.3 Compute 51-Target Adaptive Targets & AUC-ECDF Evaluation Matrix
Computes the adaptive precision target grid and evaluates the global AUC-ECDF matrix across all solvers and test conditions required by Figures 9D and 9E.

In [3]:
# ── Compute 51-Target Adaptive Targets & AUC-ECDF Matrix ───────────────────────
targets_dict = {n_std: service.compute_adaptive_targets(all_benchmark_data, noise_std=n_std) for n_std in all_noise_stds}
df_all = service.compute_auc_ecdf_matrix(all_benchmark_data, ALL_SOLVERS_ORDER, targets=targets_dict, group_by="condition")

# Compute canonical solver ranking by grouping across clean/adapted variations
df_all["Canonical Solver"] = df_all["Solver"].str.replace(r" \(noise-adapted\)", "", regex=True)
solver_overall_auc = df_all.groupby("Canonical Solver")["AUC-ECDF (%)"].mean().sort_values(ascending=False)
sorted_solvers_auc = solver_overall_auc.index.tolist()

print(f"✅ Computed AUC-ECDF matrix for {len(df_all)} condition entries across {len(sorted_solvers_auc)} canonical solvers.")


✅ Computed AUC-ECDF matrix for 2097 condition entries across 31 canonical solvers.


---
## Section 2: Core Thesis Publication Figures

### 2.1 Figure 9D (RQ1): Solver × BBOB Problem Function Performance Matrix
**Research Question 1**: *Which continuous landscape topologies favor LLM-generated heuristics versus analytical classical baselines?*
Generates a comprehensive AUC-ECDF heatmap across Sphere (f1), Rosenbrock (f8), Discus (f11), Rastrigin (f15), and Gallagher 101 Peaks (f21), saved to `results/main_results/fig_09d_auc_ecdf_by_problem.png`.

In [5]:
# ── THESIS Figure 9D: Solver × Problem Function Matrix (Heatmap) ─────────────
prob_ids = [p for p in [1, 8, 11, 15, 21] if p in PROBLEM_IDS]
prob_labels = [f"{BBOBFunction.get_name(p)} (f{p})" for p in prob_ids]
solvers_y = list(solver_overall_auc.index)

matrix_clean = np.zeros((len(solvers_y), len(prob_ids)))
matrix_noisy = np.zeros((len(solvers_y), len(prob_ids)))
for r_idx, s in enumerate(solvers_y):
    for c_idx, p in enumerate(prob_ids):
        # Clean regime
        c_sub = df_all[(df_all["Canonical Solver"] == s) & (df_all["Problem ID"] == p) & (df_all["Noise Std"] == clean_std)]
        # Noisy regime: matches either clean-transfer or noise-adapted solver
        n_sub = df_all[(df_all["Canonical Solver"] == s) & (df_all["Problem ID"] == p) & (df_all["Noise Std"] == noisy_std)]
        matrix_clean[r_idx, c_idx] = c_sub["AUC-ECDF (%)"].mean() if not c_sub.empty else 0.0
        matrix_noisy[r_idx, c_idx] = n_sub["AUC-ECDF (%)"].mean() if not n_sub.empty else 0.0

fig9d = make_subplots(
    rows=1, cols=2,
    subplot_titles=[f"<b>(A) Clean (σ={clean_std})</b>", f"<b>(B) Noisy (σ={noisy_std})</b>"],
    horizontal_spacing=0.10,
    shared_yaxes=True
)
fig9d.add_trace(go.Heatmap(
    z=matrix_clean, x=prob_labels, y=solvers_y,
    colorscale=[[0.0, "#F8FAFC"], [0.25, "#E0F2FE"], [0.5, "#BAE6FD"], [0.75, "#7DD3FC"], [1.0, "#38BDF8"]], zmin=0, zmax=70,
    text=[[f"{v:.1f}%" if v > 0 else "" for v in row] for row in matrix_clean],
    texttemplate="%{text}",
    textfont=dict(size=12, family=FONT_FAMILY),
    showscale=False,
), row=1, col=1)
fig9d.add_trace(go.Heatmap(
    z=matrix_noisy, x=prob_labels, y=solvers_y,
    colorscale=[[0.0, "#F8FAFC"], [0.25, "#E0F2FE"], [0.5, "#BAE6FD"], [0.75, "#7DD3FC"], [1.0, "#38BDF8"]], zmin=0, zmax=70,
    text=[[f"{v:.1f}%" if v > 0 else "" for v in row] for row in matrix_noisy],
    texttemplate="%{text}",
    textfont=dict(size=12, family=FONT_FAMILY),
    colorbar=dict(
        title="<b>AUC-ECDF (%)</b>",
        title_font=dict(size=14, family=FONT_FAMILY),
        title_side="top",
        tickfont=dict(size=12, family=FONT_FAMILY),
        len=0.85
    ),
), row=1, col=2)

for anno in fig9d.layout.annotations:
    anno.update(font=dict(size=16, color="#0F172A", family=FONT_FAMILY))

plot_h = max(700, len(solvers_y) * 32 + 180)
fig9d.update_layout(
    template="plotly_white",
    title=dict(
        text="<b>Figure 9D: Solver Performance Matrix Across BBOB Problem Landscapes</b><br><span style='font-size:13px;color:#475569;font-weight:normal;'>Area Under Runtime ECDF (AUC-ECDF %) Across Canonical Function Classes in Clean vs. Noisy Regimes</span>",
        font=dict(size=20, color="#0F172A", family=FONT_FAMILY),
        x=0.02, y=0.97
    ),
    width=1420, height=plot_h,
    margin=dict(l=220, r=40, t=110, b=90),
)
fig9d.update_xaxes(tickangle=-25, tickfont=dict(size=13, family=FONT_FAMILY, color="#1E293B"), row=1, col=1)
fig9d.update_xaxes(tickangle=-25, tickfont=dict(size=13, family=FONT_FAMILY, color="#1E293B"), row=1, col=2)
fig9d.update_yaxes(tickfont=dict(size=13, family=FONT_FAMILY, color="#1E293B"), autorange="reversed", row=1, col=1)

out_9d = THESIS_SUMMARY_DIR / "fig_09d_auc_ecdf_by_problem.png"
fig9d.write_image(str(out_9d), scale=3)
print("✅ Figure 9D (By Problem Heatmap) generated in results/main_results/")


2026-09-16 14:00:14 INFO TemporaryDirectory.cleanup() worked.
2026-09-16 14:00:14 INFO shutil.rmtree worked.
2026-09-16 14:00:14 INFO TemporaryDirectory.cleanup() worked.
2026-09-16 14:00:14 INFO shutil.rmtree worked.
2026-09-16 14:00:14 INFO Chromium init'ed with kwargs {}
2026-09-16 14:00:14 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-09-16 14:00:14 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp3q0oyv5t.
2026-09-16 14:00:14 INFO Opening browser.
2026-09-16 14:00:14 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp1en9wunq.
2026-09-16 14:00:14 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp1en9wunq
2026-09-16 14:00:17 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp3q0oyv5t/index.html
2026-09-16 14:00:20 INFO Getting tab from queue (has 1)
2026-09-16 14:00:20 INFO Got 6684
2026-09-16 14:00:20 INFO Reloading

✅ Figure 9D (By Problem Heatmap) generated in results/main_results/


### 2.2 Figure 9E (Sub-RQ A): LLM Parameter Scale Ablation Across Dimensions
Analyzes the empirical performance scaling trajectory across parameter scales (3B, 7B, 14B, 32B) and dimensionality regimes ($D \in \{2, 3, 5, 10\}$) in deterministic ($\sigma = 0.0$) and stochastic noise ($\sigma > 0.0$) conditions relative to classical numerical optimizers.


In [6]:
# ── THESIS Figure 9E: LLM Parameter Scale Ablation Across Dimensions ────────
fig9e = make_subplots(
    rows=1, cols=2,
    subplot_titles=[f"<b>(A) Clean (σ={clean_std})</b>", f"<b>(B) Noisy (σ={noisy_std})</b>"],
    horizontal_spacing=0.10,
    shared_yaxes=True
)
dims = [d for d in [2, 3, 5, 10] if d in all_dims]

# Discover model families dynamically
def extract_model_scale(m: str) -> float:
    match = re.search(r"(\d+(?:\.\d+)?)\s*[bB]", m)
    return float(match.group(1)) if match else 0.0

discovered_models = sorted(list(MODELS_TO_SOLVERS.keys()), key=extract_model_scale)

for col_idx, (n_std, title_sfx) in enumerate([(clean_std, "Clean"), (noisy_std, "Noisy")], start=1):
    for model_name in discovered_models:
        sub_m = df_all[df_all["Solver"].str.startswith(f"{model_name} /")]
        c_m = sub_m[sub_m["Noise Std"] == n_std]
        vals_m = [c_m[c_m["Dim"] == d]["AUC-ECDF (%)"].mean() if not c_m[c_m["Dim"] == d].empty else np.nan for d in dims]
        if any(pd.notna(v) and not np.isnan(v) for v in vals_m):
            m_color = get_model_scale_color(model_name)
            fig9e.add_trace(go.Bar(
                x=[f"{d}D" for d in dims], y=vals_m,
                name=model_name,
                marker=dict(color=m_color, line=dict(color="#0F172A", width=0.8)),
                text=[f"{v:.1f}%" if pd.notna(v) and not np.isnan(v) else "" for v in vals_m], textposition="outside",
                textfont=dict(size=11, family=FONT_FAMILY, color="#1E293B"),
                showlegend=(col_idx == 1)
            ), row=1, col=col_idx)

    for baseline in CLASSICAL_SOLVERS_ORDER:
        b_sub = df_all[(df_all["Solver"] == baseline) & (df_all["Noise Std"] == n_std)]
        b_vals = [b_sub[b_sub["Dim"] == d]["AUC-ECDF (%)"].mean() if not b_sub[b_sub["Dim"] == d].empty else np.nan for d in dims]
        b_style = get_solver_line_style(baseline)
        fig9e.add_trace(go.Scatter(
            x=[f"{d}D" for d in dims], y=b_vals,
            mode="lines+markers", name=baseline,
            line=dict(color=b_style["color"], dash=b_style["dash"], width=2.2),
            marker=dict(size=8, symbol="diamond" if "cma" in baseline.lower() else ("square" if "pso" in baseline.lower() else "circle")),
            showlegend=(col_idx == 1)
        ), row=1, col=col_idx)

for anno in fig9e.layout.annotations:
    anno.update(font=dict(size=16, color="#0F172A", family=FONT_FAMILY))

fig9e.update_layout(
    template="plotly_white",
    title=dict(
        text="<b>Figure 9E: LLM Parameter Scale Ablation Across Dimensions</b><br><span style='font-size:13px;color:#475569;font-weight:normal;'>Mean Area Under Runtime ECDF (AUC-ECDF %) Across Synthesized Model Scales vs. Classical Baselines in Clean and Noisy Regimes</span>",
        font=dict(size=20, color="#0F172A", family=FONT_FAMILY),
        x=0.02, y=0.97
    ),
    barmode="group",
    width=1380, height=640,
    margin=dict(l=70, r=40, t=110, b=90),
    legend=dict(
        orientation="h", yanchor="top", y=-0.14, xanchor="center", x=0.5,
        bgcolor="rgba(255,255,255,0.95)", bordercolor="#E2E8F0", borderwidth=1,
        font=dict(size=13, family=FONT_FAMILY)
    )
)
fig9e.update_yaxes(
    title_text="<b>Mean AUC-ECDF (%)</b>",
    title_font=dict(size=15, family=FONT_FAMILY, color="#0F172A"),
    tickfont=dict(size=13, family=FONT_FAMILY, color="#1E293B"),
    range=[0, 65], showgrid=True, gridcolor="#F1F5F9", row=1, col=1
)
fig9e.update_yaxes(
    tickfont=dict(size=13, family=FONT_FAMILY, color="#1E293B"),
    range=[0, 65], showgrid=True, gridcolor="#F1F5F9", row=1, col=2
)
fig9e.update_xaxes(
    title_text="<b>Problem Dimension</b>",
    title_font=dict(size=15, family=FONT_FAMILY, color="#0F172A"),
    tickfont=dict(size=13, family=FONT_FAMILY, color="#1E293B"),
    row=1, col=1
)
fig9e.update_xaxes(
    title_text="<b>Problem Dimension</b>",
    title_font=dict(size=15, family=FONT_FAMILY, color="#0F172A"),
    tickfont=dict(size=13, family=FONT_FAMILY, color="#1E293B"),
    row=1, col=2
)

out_9e = THESIS_SUMMARY_DIR / "fig_09e_auc_ecdf_model_scale.png"
fig9e.write_image(str(out_9e), scale=3)
print("✅ Figure 9E (Model Scale Ablation) generated in results/main_results/")


2026-09-16 14:00:29 INFO TemporaryDirectory.cleanup() worked.
2026-09-16 14:00:29 INFO shutil.rmtree worked.
2026-09-16 14:00:29 INFO Chromium init'ed with kwargs {}
2026-09-16 14:00:29 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-09-16 14:00:29 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpe7etmuv4.
2026-09-16 14:00:29 INFO Opening browser.
2026-09-16 14:00:29 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmptwqe_94y.
2026-09-16 14:00:29 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmptwqe_94y
2026-09-16 14:00:31 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpe7etmuv4/index.html
2026-09-16 14:00:32 INFO Getting tab from queue (has 1)
2026-09-16 14:00:32 INFO Got 4DB0
2026-09-16 14:00:33 INFO Reloading tab 4DB0 before return.
2026-09-16 14:00:33 INFO Putting tab 4DB0 back (queue size: 0).
2026-09-16 14:00:33 

✅ Figure 9E (Model Scale Ablation) generated in results/main_results/


### 2.3 Model-Specific Success Rates by Problem Hardness Class
Separates success rates across Separable, Conditioning, and Multi-Modal function classes for each LLM family in clean vs. noisy regimes, exporting to `results/profiles/`.

In [7]:
# ── Model-Specific Success Rate by Landscape Hardness (Clean vs. Noisy) ──
def render_model_success_rate_by_hardness(model_tag: str, solvers_list: list, dim: int):
    # Check if there is any data for this model in this dimension
    has_model_data = False
    for noise_level in [clean_std, noisy_std]:
        df_hard = service.compute_hardness_success_rates(all_benchmark_data, dim, solvers_list, noise_level=noise_level)
        if not df_hard.empty and any(" / " in s for s in df_hard["Solver"].unique()):
            has_model_data = True
            break
    if not has_model_data:
        return

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            f"<b>(A) Deterministic Landscape (σ={clean_std}, {dim}D)</b>",
            f"<b>(B) Noisy Stochastic Landscape (σ={noisy_std}, {dim}D)</b>"
        ),
        horizontal_spacing=0.10
    )
    
    for c_idx, noise_level in enumerate([clean_std, noisy_std], start=1):
        df_hard = service.compute_hardness_success_rates(all_benchmark_data, dim, solvers_list, noise_level=noise_level)
        for solver in solvers_list:
            sub_s = df_hard[df_hard["Solver"] == solver] if not df_hard.empty else pd.DataFrame()
            if not sub_s.empty:
                fig.add_trace(
                    go.Bar(
                        name=solver,
                        x=sub_s["Class"],
                        y=sub_s["Success Rate"],
                        marker=dict(
                            color=get_solver_color(solver),
                            line=dict(color="#0F172A", width=0.8)
                        ),
                        showlegend=(c_idx == 1)
                    ),
                    row=1, col=c_idx
                )
                
    fig.update_xaxes(
        title_text="<b>Landscape Hardness Class</b>",
        title_font=dict(size=14, family=FONT_FAMILY, color="#0F172A"),
        tickangle=-15,
        tickfont=dict(size=12, family=FONT_FAMILY, color="#1E293B"),
        row=1, col=1
    )
    fig.update_xaxes(
        title_text="<b>Landscape Hardness Class</b>",
        title_font=dict(size=14, family=FONT_FAMILY, color="#0F172A"),
        tickangle=-15,
        tickfont=dict(size=12, family=FONT_FAMILY, color="#1E293B"),
        row=1, col=2
    )
    fig.update_yaxes(
        title_text="<b>Target Success Rate (Δy ≤ 10⁻⁸)</b>",
        title_font=dict(size=14, family=FONT_FAMILY, color="#0F172A"),
        tickfont=dict(size=12, family=FONT_FAMILY, color="#1E293B"),
        range=[0, 1.10], showgrid=True, gridcolor="#F1F5F9", row=1, col=1
    )
    fig.update_yaxes(
        tickfont=dict(size=12, family=FONT_FAMILY, color="#1E293B"),
        range=[0, 1.10], showgrid=True, gridcolor="#F1F5F9", row=1, col=2
    )
    
    for anno in fig.layout.annotations:
        anno.update(font=dict(size=15, color="#0F172A", family=FONT_FAMILY))
        
    fig.update_layout(
        template="plotly_white",
        title=dict(
            text=f"<b>Empirical Success Rate by BBOB Landscape Hardness — {model_tag.upper()} ({dim}D)</b><br><span style='font-size:13px;color:#475569;font-weight:normal;'>Comparison of Target Precision Hitting Rates Across 5 Problem Classes in Deterministic vs. Noisy Regimes</span>",
            x=0.02, y=0.96,
            font=dict(size=16, color="#1E293B", family=FONT_FAMILY)
        ),
        barmode="group",
        bargap=0.25,
        bargroupgap=0.08,
        width=1240, height=590,
        margin=dict(l=80, r=40, t=100, b=120),
        legend=dict(
            orientation="h",
            yanchor="top", y=-0.22,
            xanchor="center", x=0.5,
            bgcolor="rgba(255,255,255,0.95)",
            bordercolor="#E2E8F0",
            borderwidth=1,
            font=dict(size=12, family=FONT_FAMILY)
        )
    )
    
    slug = resolve_canonical_model_slug(model_tag)
    m_dir = PROFILES_DIR / slug / f"{dim}D"
    m_dir.mkdir(parents=True, exist_ok=True)
    out_p = m_dir / "figure_success_rate_by_hardness.png"
    fig.write_image(str(out_p), scale=3)

for dim in all_dims:
    for model_name, solvers_list in MODELS_TO_SOLVERS.items():
        solvers_to_plot = solvers_list + CLASSICAL_SOLVERS_ORDER
        render_model_success_rate_by_hardness(model_name, solvers_to_plot, dim)

print("✅ Model-specific success rate by hardness generated for all models and dimensions.")


2026-09-16 14:00:41 INFO TemporaryDirectory.cleanup() worked.
2026-09-16 14:00:41 INFO shutil.rmtree worked.
2026-09-16 14:00:41 INFO Chromium init'ed with kwargs {}
2026-09-16 14:00:41 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-09-16 14:00:41 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpt3t980br.
2026-09-16 14:00:41 INFO Opening browser.
2026-09-16 14:00:41 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp8x26rx3h.
2026-09-16 14:00:41 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp8x26rx3h
2026-09-16 14:00:43 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpt3t980br/index.html
2026-09-16 14:00:45 INFO Getting tab from queue (has 1)
2026-09-16 14:00:45 INFO Got F148
2026-09-16 14:00:45 INFO Reloading tab F148 before return.
2026-09-16 14:00:45 INFO Putting tab F148 back (queue size: 0).
2026-09-16 14:00:45 

✅ Model-specific success rate by hardness generated for all models and dimensions.


---
## Section 3: Search Trajectories & Empirical Runtime ECDFs

### 3.1 Part 1: Explicit Noise Performance Profiles (In-Distribution / Noise-Informed vs. Baselines)
Visualizes full optimization convergence trajectories and empirical runtime ECDFs for noise-informed LLaMEA champions alongside classical baselines (CMA-ES, DE, PSO) across deterministic ($\sigma=0.0$) and noisy ($\sigma \in \{0.05, 0.1, 0.2\}$) environments, saved to `results/figures/02_explicit/`.

In [8]:
# ── THESIS Part 1: Explicit Noise Performance Profiles ─────────────────────
from benchmarking.infra.storage import EvaluationConfigRepository
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

config_repo = EvaluationConfigRepository()
bench_cfg = config_repo.load_config()
budget_multiplier = getattr(bench_cfg, "budget_multiplier", 10000)

MAX_EVAL_BUDGET = 1_000_000  # Standardized 10^6 evaluation budget
eval_grid = np.logspace(0, 6, 300)
tickvals = [1, 10, 100, 1000, 10000, 100000, 1000000]
ticktext = ['1', '10', '100', '1k', '10k', '100k', '1M']

coords = [((i // 3) + 1, (i % 3) + 1) for i in range(len(PROBLEM_IDS) + 1)]
subplot_titles = [f'<b>{BBOBFunction.get_name(p).replace(" Multi-Modal", "")}</b><br><sup>{BBOBFunction.get_class(p)}</sup>' for p in PROBLEM_IDS]
subplot_titles.append('<b>Overall Aggregate Profile</b><br><sup>Mean across 5 BBOB Problem Classes</sup>')

for dim in all_dims:
    for model_name, s_list in MODELS_TO_SOLVERS.items():
        slug = resolve_canonical_model_slug(model_name)
        m_dir = EXPLICIT_DIR / slug / f'{dim}D'

        for n_std in all_noise_stds:
            target_evals = targets_dict.get(n_std, service.compute_adaptive_targets(all_benchmark_data, noise_std=n_std))
            cond_dir = m_dir / f'std_{n_std}'
            cond_dir.mkdir(parents=True, exist_ok=True)
            conv_file = cond_dir / 'convergence_trajectories.png'
            ecdf_file = cond_dir / 'target_precision_ecdf.png'

            # Dynamically select appropriate strategy solvers for this environment
            if n_std == 0.0:
                env_solvers = [s for s in s_list if '(noise-adapted)' not in s and '(noise-implicit)' not in s]
            else:
                noisy_solvers = [s for s in s_list if '(noise-adapted)' in s]
                clean_transfer = [s for s in s_list if '(noise-adapted)' not in s and '(noise-implicit)' not in s]
                env_solvers = noisy_solvers if noisy_solvers else clean_transfer

            active_solvers = env_solvers + CLASSICAL_BASELINES
            condition_trace_dirs = [TRACES_DIR / f'{dim}D' / f'std_{n_std}' / f'f{p}' for p in PROBLEM_IDS]

            needs_conv = is_figure_outdated(conv_file, condition_trace_dirs)
            needs_ecdf = is_figure_outdated(ecdf_file, condition_trace_dirs)

            if not needs_conv and not needs_ecdf:
                continue

            clean_model_title = get_clean_model_label(slug)
            env_label = "Deterministic Clean" if n_std == 0.0 else f"Noisy Stochastic (σ={n_std})"

            # ── Convergence Trajectories ──────────────────────────────────────
            if needs_conv:
                fig_conv = make_subplots(
                    rows=2, cols=3,
                    subplot_titles=subplot_titles,
                    horizontal_spacing=0.08,
                    vertical_spacing=0.22
                )

                for p_idx, pid in enumerate(PROBLEM_IDS):
                    r, c = coords[p_idx]
                    for s in active_solvers:
                        color = service.palette.get_color(s)
                        dash_style = service.palette.get_dash(s)
                        data = service.get_convergence_trajectory(
                            all_benchmark_data,
                            dim=dim,
                            noise_std=n_std,
                            problem_id=pid,
                            solver=s,
                            eval_grid=eval_grid
                        )
                        if data is not None and not np.isnan(data["median"]).all():
                            clean_name = service.palette.clean_name(s).replace(" (noise-adapted)", "").replace(" (noise-implicit)", "")
                            med = np.maximum(data["median"], 1e-12)
                            q25 = np.maximum(data["q25"], 1e-12)
                            q75 = np.maximum(data["q75"], 1e-12)

                            fig_conv.add_trace(go.Scatter(
                                x=eval_grid, y=q75,
                                mode='lines', line=dict(width=0),
                                showlegend=False, hoverinfo='skip'
                            ), row=r, col=c)
                            fig_conv.add_trace(go.Scatter(
                                x=eval_grid, y=q25,
                                mode='lines', line=dict(width=0),
                                fill='tonexty',
                                fillcolor=service.palette.hex_to_rgba(color, 0.12),
                                showlegend=False, hoverinfo='skip'
                            ), row=r, col=c)
                            fig_conv.add_trace(go.Scatter(
                                x=eval_grid, y=med,
                                mode='lines',
                                name=clean_name,
                                line=dict(color=color, width=2.2, dash=dash_style),
                                showlegend=(p_idx == 0)
                            ), row=r, col=c)

                # Overall Aggregate Profile
                r, c = coords[-1]
                for s in active_solvers:
                    color = service.palette.get_color(s)
                    dash_style = service.palette.get_dash(s)
                    data = service.get_aggregate_convergence(
                        all_benchmark_data,
                        dim=dim,
                        noise_std=n_std,
                        solver=s,
                        eval_grid=eval_grid
                    )
                    if data is not None and not np.isnan(data["median"]).all():
                        clean_name = service.palette.clean_name(s).replace(" (noise-adapted)", "").replace(" (noise-implicit)", "")
                        med = np.maximum(data["median"], 1e-12)
                        q25 = np.maximum(data["q25"], 1e-12)
                        q75 = np.maximum(data["q75"], 1e-12)

                        fig_conv.add_trace(go.Scatter(
                            x=eval_grid, y=q75,
                            mode='lines', line=dict(width=0),
                            showlegend=False, hoverinfo='skip'
                        ), row=r, col=c)
                        fig_conv.add_trace(go.Scatter(
                            x=eval_grid, y=q25,
                            mode='lines', line=dict(width=0),
                            fill='tonexty',
                            fillcolor=service.palette.hex_to_rgba(color, 0.12),
                            showlegend=False, hoverinfo='skip'
                        ), row=r, col=c)
                        fig_conv.add_trace(go.Scatter(
                            x=eval_grid, y=med,
                            mode='lines',
                            name=clean_name,
                            line=dict(color=color, width=2.4, dash=dash_style),
                            showlegend=False
                        ), row=r, col=c)

                fig_conv.update_xaxes(
                    type="log",
                    range=[0, 6],
                    tickvals=tickvals,
                    ticktext=ticktext,
                    title_text="<b>Evaluations</b>",
                    showgrid=True,
                    gridcolor="#F1F5F9",
                    linecolor="#CBD5E1",
                    ticks="outside"
                )
                fig_conv.update_yaxes(
                    type="log",
                    title_text="<b>Mean Error Δy (Log Scale)</b>",
                    showgrid=True,
                    gridcolor="#F1F5F9",
                    linecolor="#CBD5E1",
                    ticks="outside"
                )
                for anno in fig_conv.layout.annotations:
                    anno.update(font=dict(size=14, color="#0F172A", family=FONT_FAMILY))

                fig_conv.update_layout(
                    template="plotly_white",
                    title=dict(
                        text=f"<b>Explicit Noise Performance Profiles [{env_label}] — {clean_model_title} ({dim}D)</b><br><span style='font-size:13px;color:#475569;font-weight:normal;'>Empirical Convergence Trajectories (Median ± IQR, N=20 Seeds, Budget = 1,000,000 evals)</span>",
                        font=dict(size=18, color="#0F172A", family=FONT_FAMILY),
                        x=0.02, y=0.97
                    ),
                    width=1340, height=860,
                    margin=dict(l=60, r=40, t=110, b=120),
                    legend=dict(
                        orientation="h",
                        yanchor="top",
                        y=-0.14,
                        xanchor="center",
                        x=0.5,
                        bgcolor="rgba(255, 255, 255, 0.95)",
                        bordercolor="#E2E8F0",
                        borderwidth=1,
                        font=dict(size=12, family=FONT_FAMILY)
                    )
                )
                fig_conv.write_image(str(conv_file), scale=3)

            # ── Runtime ECDF Curves ───────────────────────────────────────────
            if needs_ecdf:
                fig_ecdf = make_subplots(
                    rows=2, cols=3,
                    subplot_titles=subplot_titles,
                    horizontal_spacing=0.08,
                    vertical_spacing=0.22
                )

                for p_idx, pid in enumerate(PROBLEM_IDS):
                    r, c = coords[p_idx]
                    for s in active_solvers:
                        color = service.palette.get_color(s)
                        dash_style = service.palette.get_dash(s)
                        curve = service.get_target_ecdf_curve(
                            all_benchmark_data,
                            target_evals,
                            dim=dim,
                            noise_std=n_std,
                            problem_id=pid,
                            solver=s,
                            eval_grid=eval_grid
                        )
                        if curve is not None and not np.isnan(curve).all():
                            clean_name = service.palette.clean_name(s).replace(" (noise-adapted)", "").replace(" (noise-implicit)", "")
                            fig_ecdf.add_trace(go.Scatter(
                                x=eval_grid, y=curve,
                                mode='lines',
                                name=clean_name,
                                line=dict(color=color, width=2.2, dash=dash_style),
                                showlegend=(p_idx == 0)
                            ), row=r, col=c)

                # Overall Aggregate Profile
                r, c = coords[-1]
                for s in active_solvers:
                    color = service.palette.get_color(s)
                    dash_style = service.palette.get_dash(s)
                    curve = service.get_aggregate_target_ecdf_curve(
                        all_benchmark_data,
                        target_evals,
                        dim=dim,
                        noise_std=n_std,
                        solver=s,
                        eval_grid=eval_grid
                    )
                    if curve is not None and not np.isnan(curve).all():
                        clean_name = service.palette.clean_name(s).replace(" (noise-adapted)", "").replace(" (noise-implicit)", "")
                        fig_ecdf.add_trace(go.Scatter(
                            x=eval_grid, y=curve,
                            mode='lines',
                            name=clean_name,
                            line=dict(color=color, width=2.4, dash=dash_style),
                            showlegend=False
                        ), row=r, col=c)

                fig_ecdf.update_xaxes(
                    type="log",
                    range=[0, 6],
                    tickvals=tickvals,
                    ticktext=ticktext,
                    title_text="<b>Evaluations</b>",
                    showgrid=True,
                    gridcolor="#F1F5F9",
                    linecolor="#CBD5E1",
                    ticks="outside"
                )
                fig_ecdf.update_yaxes(
                    range=[0, 1.05],
                    title_text="<b>Proportion Solved</b>",
                    showgrid=True,
                    gridcolor="#F1F5F9",
                    linecolor="#CBD5E1",
                    ticks="outside"
                )
                for anno in fig_ecdf.layout.annotations:
                    anno.update(font=dict(size=14, color="#0F172A", family=FONT_FAMILY))

                fig_ecdf.update_layout(
                    template="plotly_white",
                    title=dict(
                        text=f"<b>Explicit Runtime ECDF [{env_label}] — {clean_model_title} ({dim}D)</b><br><span style='font-size:13px;color:#475569;font-weight:normal;'>Proportion of Targets Solved (51 Adaptive Targets in 1.0e-03 ≤ Δy ≤ 1.0e+02) vs. Evaluation Budget</span>",
                        font=dict(size=18, color="#0F172A", family=FONT_FAMILY),
                        x=0.02, y=0.97
                    ),
                    width=1340, height=860,
                    margin=dict(l=60, r=40, t=110, b=120),
                    legend=dict(
                        orientation="h",
                        yanchor="top",
                        y=-0.14,
                        xanchor="center",
                        x=0.5,
                        bgcolor="rgba(255, 255, 255, 0.95)",
                        bordercolor="#E2E8F0",
                        borderwidth=1,
                        font=dict(size=12, family=FONT_FAMILY)
                    )
                )
                fig_ecdf.write_image(str(ecdf_file), scale=3)

print("✅ Part 1: Explicit Noise Performance Profiles updated in results/figures/02_explicit/")


2026-09-16 14:01:56 INFO TemporaryDirectory.cleanup() worked.
2026-09-16 14:01:56 INFO shutil.rmtree worked.
2026-09-16 14:01:56 INFO TemporaryDirectory.cleanup() worked.
2026-09-16 14:01:56 INFO shutil.rmtree worked.
2026-09-16 14:01:57 INFO Chromium init'ed with kwargs {}
2026-09-16 14:01:57 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-09-16 14:01:57 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpd3h9_bou.
2026-09-16 14:01:57 INFO Opening browser.
2026-09-16 14:01:57 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpkziww8bj.
2026-09-16 14:01:57 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpkziww8bj
2026-09-16 14:02:01 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpd3h9_bou/index.html
2026-09-16 14:02:03 INFO Getting tab from queue (has 1)
2026-09-16 14:02:03 INFO Got 0802
2026-09-16 14:02:04 INFO Reloading

✅ 6-Panel Convergence and BBOB Runtime ECDF profiles generated successfully.


### 3.2 Part 2: Implicit Noise Performance Profiles (Noise-Blind Heuristics vs. Classical Baselines)
Visualizes full optimization convergence trajectories and empirical runtime ECDFs for noise-blind LLaMEA heuristics deployed directly on stochastic noisy landscapes ($\sigma \in \{0.05, 0.1, 0.2\}$) alongside classical baselines (CMA-ES, DE, PSO), saved to `results/figures/03_implicit/`.

In [ ]:
# ── THESIS Part 2: Implicit Noise Performance Profiles ─────────────────────
# Evaluates noise-blind LLaMEA heuristics on noisy landscapes alongside classical baselines

for dim in all_dims:
    for model_name, s_list in MODELS_TO_SOLVERS.items():
        slug = resolve_canonical_model_slug(model_name)
        m_dir = IMPLICIT_DIR / slug / f'{dim}D'

        # Only noisy environments apply for implicit adaptation evaluation
        for n_std in [s for s in all_noise_stds if s > 0.0]:
            target_evals = targets_dict.get(n_std, service.compute_adaptive_targets(all_benchmark_data, noise_std=n_std))
            cond_dir = m_dir / f'std_{n_std}'
            cond_dir.mkdir(parents=True, exist_ok=True)
            conv_file = cond_dir / 'convergence_trajectories.png'
            ecdf_file = cond_dir / 'target_precision_ecdf.png'

            # Select noise-blind (implicit) solvers for this model
            implicit_solvers = [s for s in s_list if '(noise-implicit)' in s]
            if not implicit_solvers:
                continue

            active_solvers = implicit_solvers + CLASSICAL_BASELINES
            condition_trace_dirs = [TRACES_DIR / f'{dim}D' / f'std_{n_std}' / f'f{p}' for p in PROBLEM_IDS]

            needs_conv = is_figure_outdated(conv_file, condition_trace_dirs)
            needs_ecdf = is_figure_outdated(ecdf_file, condition_trace_dirs)

            if not needs_conv and not needs_ecdf:
                continue

            clean_model_title = get_clean_model_label(slug)
            env_label = f"Noisy Stochastic (σ={n_std})"

            # ── Convergence Trajectories ──────────────────────────────────────
            if needs_conv:
                fig_conv = make_subplots(
                    rows=2, cols=3,
                    subplot_titles=subplot_titles,
                    horizontal_spacing=0.08,
                    vertical_spacing=0.22
                )

                for p_idx, pid in enumerate(PROBLEM_IDS):
                    r, c = coords[p_idx]
                    for s in active_solvers:
                        color = service.palette.get_color(s)
                        # Implicit LLaMEA champions get solid lines; classical baselines keep dashed/dotted
                        dash_style = "solid" if s in implicit_solvers else service.palette.get_dash(s)
                        data = service.get_convergence_trajectory(
                            all_benchmark_data,
                            dim=dim,
                            noise_std=n_std,
                            problem_id=pid,
                            solver=s,
                            eval_grid=eval_grid
                        )
                        if data is not None and not np.isnan(data["median"]).all():
                            clean_name = service.palette.clean_name(s).replace(" (noise-adapted)", "").replace(" (noise-implicit)", "")
                            med = np.maximum(data["median"], 1e-12)
                            q25 = np.maximum(data["q25"], 1e-12)
                            q75 = np.maximum(data["q75"], 1e-12)

                            fig_conv.add_trace(go.Scatter(
                                x=eval_grid, y=q75,
                                mode='lines', line=dict(width=0),
                                showlegend=False, hoverinfo='skip'
                            ), row=r, col=c)
                            fig_conv.add_trace(go.Scatter(
                                x=eval_grid, y=q25,
                                mode='lines', line=dict(width=0),
                                fill='tonexty',
                                fillcolor=service.palette.hex_to_rgba(color, 0.12),
                                showlegend=False, hoverinfo='skip'
                            ), row=r, col=c)
                            fig_conv.add_trace(go.Scatter(
                                x=eval_grid, y=med,
                                mode='lines',
                                name=clean_name,
                                line=dict(color=color, width=2.2, dash=dash_style),
                                showlegend=(p_idx == 0)
                            ), row=r, col=c)

                # Overall Aggregate Profile
                r, c = coords[-1]
                for s in active_solvers:
                    color = service.palette.get_color(s)
                    dash_style = "solid" if s in implicit_solvers else service.palette.get_dash(s)
                    data = service.get_aggregate_convergence(
                        all_benchmark_data,
                        dim=dim,
                        noise_std=n_std,
                        solver=s,
                        eval_grid=eval_grid
                    )
                    if data is not None and not np.isnan(data["median"]).all():
                        clean_name = service.palette.clean_name(s).replace(" (noise-adapted)", "").replace(" (noise-implicit)", "")
                        med = np.maximum(data["median"], 1e-12)
                        q25 = np.maximum(data["q25"], 1e-12)
                        q75 = np.maximum(data["q75"], 1e-12)

                        fig_conv.add_trace(go.Scatter(
                            x=eval_grid, y=q75,
                            mode='lines', line=dict(width=0),
                            showlegend=False, hoverinfo='skip'
                        ), row=r, col=c)
                        fig_conv.add_trace(go.Scatter(
                            x=eval_grid, y=q25,
                            mode='lines', line=dict(width=0),
                            fill='tonexty',
                            fillcolor=service.palette.hex_to_rgba(color, 0.12),
                            showlegend=False, hoverinfo='skip'
                        ), row=r, col=c)
                        fig_conv.add_trace(go.Scatter(
                            x=eval_grid, y=med,
                            mode='lines',
                            name=clean_name,
                            line=dict(color=color, width=2.4, dash=dash_style),
                            showlegend=False
                        ), row=r, col=c)

                fig_conv.update_xaxes(
                    type="log",
                    range=[0, 6],
                    tickvals=tickvals,
                    ticktext=ticktext,
                    title_text="<b>Evaluations</b>",
                    showgrid=True,
                    gridcolor="#F1F5F9",
                    linecolor="#CBD5E1",
                    ticks="outside"
                )
                fig_conv.update_yaxes(
                    type="log",
                    title_text="<b>Mean Error Δy (Log Scale)</b>",
                    showgrid=True,
                    gridcolor="#F1F5F9",
                    linecolor="#CBD5E1",
                    ticks="outside"
                )
                for anno in fig_conv.layout.annotations:
                    anno.update(font=dict(size=14, color="#0F172A", family=FONT_FAMILY))

                fig_conv.update_layout(
                    template="plotly_white",
                    title=dict(
                        text=f"<b>Implicit Noise Performance Profiles [{env_label}] — {clean_model_title} ({dim}D)</b><br><span style='font-size:13px;color:#475569;font-weight:normal;'>Empirical Convergence Trajectories of Noise-Blind Heuristics on Noisy Topologies vs. Classical Baselines (Budget = 1,000,000 evals)</span>",
                        font=dict(size=18, color="#0F172A", family=FONT_FAMILY),
                        x=0.02, y=0.97
                    ),
                    width=1340, height=860,
                    margin=dict(l=60, r=40, t=110, b=120),
                    legend=dict(
                        orientation="h",
                        yanchor="top",
                        y=-0.14,
                        xanchor="center",
                        x=0.5,
                        bgcolor="rgba(255, 255, 255, 0.95)",
                        bordercolor="#E2E8F0",
                        borderwidth=1,
                        font=dict(size=12, family=FONT_FAMILY)
                    )
                )
                fig_conv.write_image(str(conv_file), scale=3)

            # ── Runtime ECDF Curves ───────────────────────────────────────────
            if needs_ecdf:
                fig_ecdf = make_subplots(
                    rows=2, cols=3,
                    subplot_titles=subplot_titles,
                    horizontal_spacing=0.08,
                    vertical_spacing=0.22
                )

                for p_idx, pid in enumerate(PROBLEM_IDS):
                    r, c = coords[p_idx]
                    for s in active_solvers:
                        color = service.palette.get_color(s)
                        dash_style = "solid" if s in implicit_solvers else service.palette.get_dash(s)
                        curve = service.get_target_ecdf_curve(
                            all_benchmark_data,
                            target_evals,
                            dim=dim,
                            noise_std=n_std,
                            problem_id=pid,
                            solver=s,
                            eval_grid=eval_grid
                        )
                        if curve is not None and not np.isnan(curve).all():
                            clean_name = service.palette.clean_name(s).replace(" (noise-adapted)", "").replace(" (noise-implicit)", "")
                            fig_ecdf.add_trace(go.Scatter(
                                x=eval_grid, y=curve,
                                mode='lines',
                                name=clean_name,
                                line=dict(color=color, width=2.2, dash=dash_style),
                                showlegend=(p_idx == 0)
                            ), row=r, col=c)

                # Overall Aggregate Profile
                r, c = coords[-1]
                for s in active_solvers:
                    color = service.palette.get_color(s)
                    dash_style = "solid" if s in implicit_solvers else service.palette.get_dash(s)
                    curve = service.get_aggregate_target_ecdf_curve(
                        all_benchmark_data,
                        target_evals,
                        dim=dim,
                        noise_std=n_std,
                        solver=s,
                        eval_grid=eval_grid
                    )
                    if curve is not None and not np.isnan(curve).all():
                        clean_name = service.palette.clean_name(s).replace(" (noise-adapted)", "").replace(" (noise-implicit)", "")
                        fig_ecdf.add_trace(go.Scatter(
                            x=eval_grid, y=curve,
                            mode='lines',
                            name=clean_name,
                            line=dict(color=color, width=2.4, dash=dash_style),
                            showlegend=False
                        ), row=r, col=c)

                fig_ecdf.update_xaxes(
                    type="log",
                    range=[0, 6],
                    tickvals=tickvals,
                    ticktext=ticktext,
                    title_text="<b>Evaluations</b>",
                    showgrid=True,
                    gridcolor="#F1F5F9",
                    linecolor="#CBD5E1",
                    ticks="outside"
                )
                fig_ecdf.update_yaxes(
                    range=[0, 1.05],
                    title_text="<b>Proportion Solved</b>",
                    showgrid=True,
                    gridcolor="#F1F5F9",
                    linecolor="#CBD5E1",
                    ticks="outside"
                )
                for anno in fig_ecdf.layout.annotations:
                    anno.update(font=dict(size=14, color="#0F172A", family=FONT_FAMILY))

                fig_ecdf.update_layout(
                    template="plotly_white",
                    title=dict(
                        text=f"<b>Implicit Runtime ECDF [{env_label}] — {clean_model_title} ({dim}D)</b><br><span style='font-size:13px;color:#475569;font-weight:normal;'>Proportion of Targets Solved (51 Adaptive Targets in 1.0e-03 ≤ Δy ≤ 1.0e+02) vs. Evaluation Budget for Noise-Blind Heuristics</span>",
                        font=dict(size=18, color="#0F172A", family=FONT_FAMILY),
                        x=0.02, y=0.97
                    ),
                    width=1340, height=860,
                    margin=dict(l=60, r=40, t=110, b=120),
                    legend=dict(
                        orientation="h",
                        yanchor="top",
                        y=-0.14,
                        xanchor="center",
                        x=0.5,
                        bgcolor="rgba(255, 255, 255, 0.95)",
                        bordercolor="#E2E8F0",
                        borderwidth=1,
                        font=dict(size=12, family=FONT_FAMILY)
                    )
                )
                fig_ecdf.write_image(str(ecdf_file), scale=3)

print("✅ Part 2: Implicit Noise Performance Profiles updated in results/figures/03_implicit/")


### 3.3 Part 3: Implicit vs. Explicit Noise Adaptation (Head-to-Head Comparison)
Direct head-to-head comparison evaluating whether explicit noise awareness during synthesis provides a measurable advantage over blind heuristic synthesis on noisy landscapes:
- **Solid Lines**: Noise-Informed LLaMEA Champions (`(noise-informed)`)
- **Dashed Lines**: Noise-Blind LLaMEA Heuristics (`(noise-blind)`)
- Saved to `results/figures/04_implicit_vs_explicit/`.

In [ ]:
# ── THESIS Part 3: Implicit vs. Explicit Head-to-Head Comparison ───────────
PROMPT_COLORS = {
    "baseline": "#F59E0B",       # Amber
    "guided": "#0284C7",         # Cerulean
    "thinking": "#10B981",       # Emerald
    "vectorization": "#EF4444",  # Crimson
}

for dim in all_dims:
    for model_name, s_list in MODELS_TO_SOLVERS.items():
        slug = resolve_canonical_model_slug(model_name)
        m_dir = IMPLICIT_VS_EXPLICIT_DIR / slug / f'{dim}D'

        for n_std in [s for s in all_noise_stds if s > 0.0]:
            target_evals = targets_dict.get(n_std, service.compute_adaptive_targets(all_benchmark_data, noise_std=n_std))
            cond_dir = m_dir / f'std_{n_std}'
            cond_dir.mkdir(parents=True, exist_ok=True)
            conv_file = cond_dir / 'implicit_vs_noisy_convergence.png'
            ecdf_file = cond_dir / 'target_precision_ecdf.png'

            noisy_solvers = [s for s in s_list if '(noise-adapted)' in s]
            implicit_solvers = [s for s in s_list if '(noise-implicit)' in s]
            if not noisy_solvers or not implicit_solvers:
                continue

            paired_solvers = []
            for ns in noisy_solvers:
                strat = ns.split('/')[-1].replace('(noise-adapted)', '').strip()
                paired_solvers.append((ns, strat, 'solid', f'{strat.title()} (noise-informed)'))
            for is_s in implicit_solvers:
                strat = is_s.split('/')[-1].replace('(noise-implicit)', '').strip()
                paired_solvers.append((is_s, strat, 'dash', f'{strat.title()} (noise-blind)'))

            condition_trace_dirs = [TRACES_DIR / f'{dim}D' / f'std_{n_std}' / f'f{p}' for p in PROBLEM_IDS]
            needs_conv = is_figure_outdated(conv_file, condition_trace_dirs)
            needs_ecdf = is_figure_outdated(ecdf_file, condition_trace_dirs)

            if not needs_conv and not needs_ecdf:
                continue

            clean_model_title = get_clean_model_label(slug)

            # ── Convergence ──────────────────────────────────────────────────
            if needs_conv:
                fig_conv = make_subplots(
                    rows=2, cols=3,
                    subplot_titles=subplot_titles,
                    horizontal_spacing=0.08, vertical_spacing=0.22
                )
                for p_idx, pid in enumerate(PROBLEM_IDS):
                    r, c = coords[p_idx]
                    for s, strat, dash, lbl in paired_solvers:
                        col = PROMPT_COLORS.get(strat, '#64748B')
                        data = service.get_convergence_trajectory(
                            all_benchmark_data, dim=dim, noise_std=n_std, problem_id=pid, solver=s, eval_grid=eval_grid
                        )
                        if data is not None and not np.isnan(data["median"]).all():
                            med = np.maximum(data["median"], 1e-12)
                            q25 = np.maximum(data["q25"], 1e-12)
                            q75 = np.maximum(data["q75"], 1e-12)

                            fig_conv.add_trace(go.Scatter(
                                x=eval_grid, y=q75, mode='lines', line=dict(width=0), showlegend=False, hoverinfo='skip'
                            ), row=r, col=c)
                            fig_conv.add_trace(go.Scatter(
                                x=eval_grid, y=q25, mode='lines', line=dict(width=0), fill='tonexty',
                                fillcolor=service.palette.hex_to_rgba(col, 0.10 if dash == 'solid' else 0.05),
                                showlegend=False, hoverinfo='skip'
                            ), row=r, col=c)
                            fig_conv.add_trace(go.Scatter(
                                x=eval_grid, y=med, mode='lines', name=lbl,
                                line=dict(color=col, width=2.2 if dash == 'solid' else 1.8, dash=dash),
                                showlegend=(p_idx == 0)
                            ), row=r, col=c)

                # Overall Aggregate
                r, c = coords[-1]
                for s, strat, dash, lbl in paired_solvers:
                    col = PROMPT_COLORS.get(strat, '#64748B')
                    data = service.get_aggregate_convergence(
                        all_benchmark_data, dim=dim, noise_std=n_std, solver=s, eval_grid=eval_grid
                    )
                    if data is not None and not np.isnan(data["median"]).all():
                        med = np.maximum(data["median"], 1e-12)
                        q25 = np.maximum(data["q25"], 1e-12)
                        q75 = np.maximum(data["q75"], 1e-12)
                        fig_conv.add_trace(go.Scatter(x=eval_grid, y=q75, mode='lines', line=dict(width=0), showlegend=False), row=r, col=c)
                        fig_conv.add_trace(go.Scatter(
                            x=eval_grid, y=q25, mode='lines', line=dict(width=0), fill='tonexty',
                            fillcolor=service.palette.hex_to_rgba(col, 0.10 if dash == 'solid' else 0.05),
                            showlegend=False
                        ), row=r, col=c)
                        fig_conv.add_trace(go.Scatter(
                            x=eval_grid, y=med, mode='lines', name=lbl,
                            line=dict(color=col, width=2.4 if dash == 'solid' else 1.8, dash=dash),
                            showlegend=False
                        ), row=r, col=c)

                fig_conv.update_xaxes(type="log", range=[0, 6], tickvals=tickvals, ticktext=ticktext, title_text="<b>Evaluations</b>", showgrid=True, gridcolor="#F1F5F9", linecolor="#CBD5E1", ticks="outside")
                fig_conv.update_yaxes(type="log", title_text="<b>Mean Error Δy (Log Scale)</b>", showgrid=True, gridcolor="#F1F5F9", linecolor="#CBD5E1", ticks="outside")
                for anno in fig_conv.layout.annotations:
                    anno.update(font=dict(size=14, color="#0F172A", family=FONT_FAMILY))

                fig_conv.update_layout(
                    template="plotly_white",
                    title=dict(
                        text=f"<b>Implicit vs. Explicit Noise Adaptation [Noisy Stochastic (σ={n_std})] — {clean_model_title} ({dim}D)</b><br><span style='font-size:13px;color:#475569;font-weight:normal;'>Head-to-Head Comparison: Noise-Informed (Solid) vs. Noise-Blind (Dashed) LLaMEA Strategies (Budget = 1,000,000 evals)</span>",
                        font=dict(size=18, color="#0F172A", family=FONT_FAMILY),
                        x=0.02, y=0.97
                    ),
                    width=1340, height=860,
                    margin=dict(l=60, r=40, t=110, b=120),
                    legend=dict(
                        orientation="h", yanchor="top", y=-0.14, xanchor="center", x=0.5,
                        bgcolor="rgba(255, 255, 255, 0.95)", bordercolor="#E2E8F0", borderwidth=1,
                        font=dict(size=12, family=FONT_FAMILY)
                    )
                )
                fig_conv.write_image(str(conv_file), scale=3)

            # ── Runtime ECDF Curves ───────────────────────────────────────────
            if needs_ecdf:
                fig_ecdf = make_subplots(
                    rows=2, cols=3,
                    subplot_titles=subplot_titles,
                    horizontal_spacing=0.08, vertical_spacing=0.22
                )
                for p_idx, pid in enumerate(PROBLEM_IDS):
                    r, c = coords[p_idx]
                    for s, strat, dash, lbl in paired_solvers:
                        col = PROMPT_COLORS.get(strat, '#64748B')
                        curve = service.get_target_ecdf_curve(
                            all_benchmark_data, target_evals, dim=dim, noise_std=n_std, problem_id=pid, solver=s, eval_grid=eval_grid
                        )
                        if curve is not None and not np.isnan(curve).all():
                            fig_ecdf.add_trace(go.Scatter(
                                x=eval_grid, y=curve, mode='lines', name=lbl,
                                line=dict(color=col, width=2.2 if dash == 'solid' else 1.8, dash=dash),
                                showlegend=(p_idx == 0)
                            ), row=r, col=c)

                # Overall Aggregate
                r, c = coords[-1]
                for s, strat, dash, lbl in paired_solvers:
                    col = PROMPT_COLORS.get(strat, '#64748B')
                    curve = service.get_aggregate_target_ecdf_curve(
                        all_benchmark_data, target_evals, dim=dim, noise_std=n_std, solver=s, eval_grid=eval_grid
                    )
                    if curve is not None and not np.isnan(curve).all():
                        fig_ecdf.add_trace(go.Scatter(
                            x=eval_grid, y=curve, mode='lines', name=lbl,
                            line=dict(color=col, width=2.4 if dash == 'solid' else 1.8, dash=dash),
                            showlegend=False
                        ), row=r, col=c)

                fig_ecdf.update_xaxes(type="log", range=[0, 6], tickvals=tickvals, ticktext=ticktext, title_text="<b>Evaluations</b>", showgrid=True, gridcolor="#F1F5F9", linecolor="#CBD5E1", ticks="outside")
                fig_ecdf.update_yaxes(range=[0, 1.05], title_text="<b>Proportion Solved</b>", showgrid=True, gridcolor="#F1F5F9", linecolor="#CBD5E1", ticks="outside")
                for anno in fig_ecdf.layout.annotations:
                    anno.update(font=dict(size=14, color="#0F172A", family=FONT_FAMILY))

                fig_ecdf.update_layout(
                    template="plotly_white",
                    title=dict(
                        text=f"<b>Implicit vs. Explicit Runtime ECDF [Noisy Stochastic (σ={n_std})] — {clean_model_title} ({dim}D)</b><br><span style='font-size:13px;color:#475569;font-weight:normal;'>Proportion of Targets Solved (51 Adaptive Targets in 1.0e-03 ≤ Δy ≤ 1.0e+02) vs. Evaluation Budget: Noise-Informed (Solid) vs. Noise-Blind (Dashed)</span>",
                        font=dict(size=18, color="#0F172A", family=FONT_FAMILY),
                        x=0.02, y=0.97
                    ),
                    width=1340, height=860,
                    margin=dict(l=60, r=40, t=110, b=120),
                    legend=dict(
                        orientation="h", yanchor="top", y=-0.14, xanchor="center", x=0.5,
                        bgcolor="rgba(255, 255, 255, 0.95)", bordercolor="#E2E8F0", borderwidth=1,
                        font=dict(size=12, family=FONT_FAMILY)
                    )
                )
                fig_ecdf.write_image(str(ecdf_file), scale=3)

print("✅ Part 3: Implicit vs. Explicit Comparison updated in results/figures/04_implicit_vs_explicit/")


### 3.4 Part 4: Single-Algorithm Cross-Environment Direct Noise Overlays
Direct overlay curves comparing an individual algorithm's behavior across noise levels ($\sigma \in \{0.0, 0.05, 0.1, 0.2\}$) to isolate noise degradation and search path deviation, saved to `results/figures/05_cross_evaluation/` partitioned into `classical_baselines/`, `explicit/`, and `implicit/`.

In [ ]:
# ── THESIS Part 4: Single-Algorithm Cross-Environment Direct Noise Overlays ─
eval_grid = np.logspace(0, 6, 300)
tickvals = [1, 10, 100, 1000, 10000, 100000, 1000000]
ticktext = ['1', '10', '100', '1k', '10k', '100k', '1M']
coords = [((i // 3) + 1, (i % 3) + 1) for i in range(len(PROBLEM_IDS) + 1)]
subplot_titles = [f'<b>{BBOBFunction.get_name(p).replace(" Multi-Modal", "")}</b><br><sup>{BBOBFunction.get_class(p)}</sup>' for p in PROBLEM_IDS]
subplot_titles.append('<b>Overall Aggregate Profile</b><br><sup>Mean across 5 BBOB Problem Classes</sup>')

# Primary algorithms to analyze individually
primary_solvers = [s for s in all_benchmark_data.solvers]
NOISE_COLORS = {
    0.0: "#0284C7",   # Blue for clean
    0.05: "#10B981",  # Green for low noise
    0.1: "#F59E0B",   # Amber for medium noise
    0.2: "#EF4444"    # Red for high noise
}

for dim in all_dims:
    for solver in primary_solvers:
        is_classical = any(c in solver.lower() for c in ["cma", "pso", "de"])
        clean_name = service.palette.clean_name(solver)

        if is_classical:
            s_slug = solver.lower().replace("-", "_")
            out_dir = CROSS_EVAL_DIR / f"{dim}D" / "classical_baselines" / s_slug
        elif "(noise-implicit)" in solver:
            parts = solver.replace(" (noise-implicit)", "").split(" / ")
            model_slug = parts[0].lower().replace(" ", "-")
            strat_slug = parts[1].lower() if len(parts) > 1 else "default"
            out_dir = CROSS_EVAL_DIR / f"{dim}D" / "implicit" / model_slug / strat_slug
        else:
            parts = solver.replace(" (noise-adapted)", "").split(" / ")
            model_slug = parts[0].lower().replace(" ", "-")
            strat_slug = parts[1].lower() if len(parts) > 1 else "default"
            out_dir = CROSS_EVAL_DIR / f"{dim}D" / "explicit" / model_slug / strat_slug

        out_dir.mkdir(parents=True, exist_ok=True)
        conv_file = out_dir / "convergence_noise_overlay.png"
        ecdf_file = out_dir / "ecdf_noise_overlay.png"

        cond_trace_dirs = [TRACES_DIR / f"{dim}D" / f"std_{n}" / f"f{p}" for n in all_noise_stds for p in PROBLEM_IDS]
        if not is_figure_outdated(conv_file, cond_trace_dirs) and not is_figure_outdated(ecdf_file, cond_trace_dirs):
            continue

        # ── Convergence Noise Overlay ──────────────────────────────────────
        fig_conv = make_subplots(
            rows=2, cols=3,
            subplot_titles=subplot_titles,
            horizontal_spacing=0.08, vertical_spacing=0.22
        )
        for p_idx, pid in enumerate(PROBLEM_IDS):
            r, c = coords[p_idx]
            for n_std in sorted(all_noise_stds):
                target_evals = targets_dict.get(n_std, service.compute_adaptive_targets(all_benchmark_data, noise_std=n_std))
                color = NOISE_COLORS.get(n_std, "#64748B")
                lbl = f"σ = {n_std}" if n_std > 0 else "Clean (σ = 0.0)"
                data = service.get_convergence_trajectory(
                    all_benchmark_data, dim=dim, noise_std=n_std, problem_id=pid, solver=solver, eval_grid=eval_grid
                )
                if data is not None and not np.isnan(data["median"]).all():
                    med = np.maximum(data["median"], 1e-12)
                    q25 = np.maximum(data["q25"], 1e-12)
                    q75 = np.maximum(data["q75"], 1e-12)

                    fig_conv.add_trace(go.Scatter(x=eval_grid, y=q75, mode='lines', line=dict(width=0), showlegend=False), row=r, col=c)
                    fig_conv.add_trace(go.Scatter(x=eval_grid, y=q25, mode='lines', line=dict(width=0), fill='tonexty', fillcolor=service.palette.hex_to_rgba(color, 0.12), showlegend=False), row=r, col=c)
                    fig_conv.add_trace(go.Scatter(
                        x=eval_grid, y=med, mode='lines', name=lbl,
                        line=dict(color=color, width=2.2), showlegend=(p_idx == 0)
                    ), row=r, col=c)

        # Aggregate
        r, c = coords[-1]
        for n_std in sorted(all_noise_stds):
            target_evals = targets_dict.get(n_std, service.compute_adaptive_targets(all_benchmark_data, noise_std=n_std))
            color = NOISE_COLORS.get(n_std, "#64748B")
            lbl = f"σ = {n_std}" if n_std > 0 else "Clean (σ = 0.0)"
            data = service.get_aggregate_convergence(
                all_benchmark_data, dim=dim, noise_std=n_std, solver=solver, eval_grid=eval_grid
            )
            if data is not None and not np.isnan(data["median"]).all():
                med = np.maximum(data["median"], 1e-12)
                q25 = np.maximum(data["q25"], 1e-12)
                q75 = np.maximum(data["q75"], 1e-12)
                fig_conv.add_trace(go.Scatter(x=eval_grid, y=q75, mode='lines', line=dict(width=0), showlegend=False), row=r, col=c)
                fig_conv.add_trace(go.Scatter(x=eval_grid, y=q25, mode='lines', line=dict(width=0), fill='tonexty', fillcolor=service.palette.hex_to_rgba(color, 0.12), showlegend=False), row=r, col=c)
                fig_conv.add_trace(go.Scatter(x=eval_grid, y=med, mode='lines', name=lbl, line=dict(color=color, width=2.4), showlegend=False), row=r, col=c)

        fig_conv.update_xaxes(type="log", range=[0, 6], tickvals=tickvals, ticktext=ticktext, title_text="<b>Evaluations</b>", showgrid=True, gridcolor="#F1F5F9", linecolor="#CBD5E1", ticks="outside")
        fig_conv.update_yaxes(type="log", title_text="<b>Mean Error Δy (Log Scale)</b>", showgrid=True, gridcolor="#F1F5F9", linecolor="#CBD5E1", ticks="outside")
        for anno in fig_conv.layout.annotations:
            anno.update(font=dict(size=14, color="#0F172A", family=FONT_FAMILY))

        fig_conv.update_layout(
            template="plotly_white",
            title=dict(
                text=f"<b>Noise Resilience Profile across σ ∈ [0.0, 0.2] — {clean_name} ({dim}D)</b><br><span style='font-size:13px;color:#475569;font-weight:normal;'>Empirical Convergence Degradation under Increasing Stochastic Noise Perturbation</span>",
                font=dict(size=18, color="#0F172A", family=FONT_FAMILY),
                x=0.02, y=0.97
            ),
            width=1340, height=860,
            margin=dict(l=60, r=40, t=110, b=120),
            legend=dict(
                orientation="h", yanchor="top", y=-0.14, xanchor="center", x=0.5,
                bgcolor="rgba(255, 255, 255, 0.95)", bordercolor="#E2E8F0", borderwidth=1,
                font=dict(size=12, family=FONT_FAMILY)
            )
        )
        fig_conv.write_image(str(conv_file), scale=3)

        # ── ECDF Noise Overlay ─────────────────────────────────────────────
        fig_ecdf = make_subplots(
            rows=2, cols=3,
            subplot_titles=subplot_titles,
            horizontal_spacing=0.08, vertical_spacing=0.22
        )
        for p_idx, pid in enumerate(PROBLEM_IDS):
            r, c = coords[p_idx]
            for n_std in sorted(all_noise_stds):
                target_evals = targets_dict.get(n_std, service.compute_adaptive_targets(all_benchmark_data, noise_std=n_std))
                color = NOISE_COLORS.get(n_std, "#64748B")
                lbl = f"σ = {n_std}" if n_std > 0 else "Clean (σ = 0.0)"
                curve = service.get_target_ecdf_curve(
                    all_benchmark_data, target_evals, dim=dim, noise_std=n_std, problem_id=pid, solver=solver, eval_grid=eval_grid
                )
                if curve is not None and not np.isnan(curve).all():
                    fig_ecdf.add_trace(go.Scatter(
                        x=eval_grid, y=curve, mode='lines', name=lbl,
                        line=dict(color=color, width=2.2), showlegend=(p_idx == 0)
                    ), row=r, col=c)

        # Aggregate ECDF
        r, c = coords[-1]
        for n_std in sorted(all_noise_stds):
            target_evals = targets_dict.get(n_std, service.compute_adaptive_targets(all_benchmark_data, noise_std=n_std))
            color = NOISE_COLORS.get(n_std, "#64748B")
            lbl = f"σ = {n_std}" if n_std > 0 else "Clean (σ = 0.0)"
            curve = service.get_aggregate_target_ecdf_curve(
                all_benchmark_data, target_evals, dim=dim, noise_std=n_std, solver=solver, eval_grid=eval_grid
            )
            if curve is not None and not np.isnan(curve).all():
                fig_ecdf.add_trace(go.Scatter(x=eval_grid, y=curve, mode='lines', name=lbl, line=dict(color=color, width=2.4), showlegend=False), row=r, col=c)

        fig_ecdf.update_xaxes(type="log", range=[0, 6], tickvals=tickvals, ticktext=ticktext, title_text="<b>Evaluations</b>", showgrid=True, gridcolor="#F1F5F9", linecolor="#CBD5E1", ticks="outside")
        fig_ecdf.update_yaxes(range=[0, 1.05], title_text="<b>Proportion Solved</b>", showgrid=True, gridcolor="#F1F5F9", linecolor="#CBD5E1", ticks="outside")
        for anno in fig_ecdf.layout.annotations:
            anno.update(font=dict(size=14, color="#0F172A", family=FONT_FAMILY))

        fig_ecdf.update_layout(
            template="plotly_white",
            title=dict(
                text=f"<b>Runtime ECDF Degradation Across Noise Levels — {clean_name} ({dim}D)</b><br><span style='font-size:13px;color:#475569;font-weight:normal;'>Proportion of Solved Targets across σ ∈ [0.0, 0.2]</span>",
                font=dict(size=18, color="#0F172A", family=FONT_FAMILY),
                x=0.02, y=0.97
            ),
            width=1340, height=860,
            margin=dict(l=60, r=40, t=110, b=120),
            legend=dict(
                orientation="h", yanchor="top", y=-0.14, xanchor="center", x=0.5,
                bgcolor="rgba(255, 255, 255, 0.95)", bordercolor="#E2E8F0", borderwidth=1,
                font=dict(size=12, family=FONT_FAMILY)
            )
        )
        fig_ecdf.write_image(str(ecdf_file), scale=3)

print("✅ Part 4: Single-Algorithm Cross-Environment Noise Overlays updated in results/figures/05_cross_evaluation/")
